# PY500 — A/B Testing, Design of Experiments, and ANOVA

A/B testing is how tech companies make **data-driven decisions** — from button colours to pricing strategies. ANOVA extends this to comparing 3+ groups simultaneously.

## Topics Covered
1. A/B Test design and analysis
2. Sample size calculation (power analysis)
3. Peeking problem and sequential testing
4. One-Way ANOVA (Analysis of Variance)
5. Post-hoc tests (Tukey HSD)
6. Repeated measures and design of experiments
7. Key libraries and packages

> **Perspective:** A/B testing is not just a statistical technique — it is a *decision framework*. The hardest part is not the math (SciPy does that). The hardest parts are: defining the right metric, estimating the minimum detectable effect, and resisting the urge to peek at results early.

---
## 1. A/B Test — Full Example

In [ ]:
## A/B Test: Does a new landing page increase conversion rate?

import numpy as np
from scipy import stats

np.random.seed(42)

# Simulate: Control (old page) vs Treatment (new page)
n_control = 5000
n_treatment = 5000

# Control: 10% conversion, Treatment: 11.5% (1.5 percentage point lift)
control = np.random.binomial(1, 0.10, n_control)
treatment = np.random.binomial(1, 0.115, n_treatment)

p_control = control.mean()
p_treatment = treatment.mean()
lift = (p_treatment - p_control) / p_control * 100

print(f"Control conversion   : {p_control:.4f} ({p_control*100:.2f}%)")
print(f"Treatment conversion : {p_treatment:.4f} ({p_treatment*100:.2f}%)")
print(f"Absolute lift        : {(p_treatment - p_control)*100:.2f} pp")
print(f"Relative lift        : {lift:.1f}%")

In [ ]:
## Statistical significance — two-proportion Z-test

# Pooled proportion under H₀ (no difference)
p_pool = (control.sum() + treatment.sum()) / (n_control + n_treatment)
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_control + 1/n_treatment))

z_stat = (p_treatment - p_control) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))   # Two-tailed

# Confidence interval for the difference
se_diff = np.sqrt(p_control*(1-p_control)/n_control + p_treatment*(1-p_treatment)/n_treatment)
ci_lower = (p_treatment - p_control) - 1.96 * se_diff
ci_upper = (p_treatment - p_control) + 1.96 * se_diff

print(f"Z-statistic : {z_stat:.3f}")
print(f"p-value     : {p_value:.4f}")
print(f"95% CI diff : [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]")
print(f"\nDecision: {'SIGNIFICANT — ship the new page!' if p_value < 0.05 else 'NOT significant — keep the old page'}")

---
## 2. Sample Size Calculation (Power Analysis)

Before running an A/B test, calculate how many users you need to detect a meaningful effect.

Inputs:
- **α** (significance level) — typically 0.05
- **Power (1-β)** — typically 0.80
- **Minimum Detectable Effect (MDE)** — smallest lift you care about

In [ ]:
## Power analysis for sample size

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# Baseline conversion = 10%, want to detect a lift to 12% (2pp lift)
baseline = 0.10
target = 0.12

effect_size = proportion_effectsize(baseline, target)
power_analysis = NormalIndPower()
sample_size = power_analysis.solve_power(
    effect_size=effect_size,
    alpha=0.05,
    power=0.80,
    alternative='two-sided'
)

print(f"Baseline rate   : {baseline*100}%")
print(f"Target rate     : {target*100}%")
print(f"Effect size (h) : {effect_size:.4f}")
print(f"Required n/group: {int(np.ceil(sample_size))}")
print(f"Total users     : {int(np.ceil(sample_size)) * 2}")

# Rule of thumb: smaller effects need larger samples.
# Detecting a 0.5pp lift (10% → 10.5%) needs ~60,000 per group!

---
## 3. The Peeking Problem

**Peeking** = checking A/B test results before reaching the planned sample size, then stopping early when results look significant.

**Why it is dangerous:** If you check daily, by random chance you will see a "significant" p-value ~30% of the time even when there is NO real effect. This dramatically inflates your false positive rate.

### Solutions
| Approach | How It Works |
|---|---|
| **Fixed horizon** | Decide sample size upfront, only look once at the end |
| **Sequential testing** | Use adjusted boundaries (O'Brien-Fleming, Pocock) |
| **Bayesian A/B testing** | Continuously update beliefs using Bayes' theorem |
| **Always-valid p-values** | Use methods that are valid at any stopping time |

In [ ]:
## Demonstrating the peeking problem

import matplotlib.pyplot as plt

np.random.seed(42)

# Both groups have SAME conversion rate (no real effect)
n_total = 5000
group_a = np.random.binomial(1, 0.10, n_total)
group_b = np.random.binomial(1, 0.10, n_total)   # Same rate!

# Check p-value at every 100 users
check_points = range(100, n_total + 1, 100)
p_values = []

for n in check_points:
    a = group_a[:n]
    b = group_b[:n]
    p_pool = (a.sum() + b.sum()) / (2 * n)
    if p_pool == 0 or p_pool == 1:
        p_values.append(1.0)
        continue
    se = np.sqrt(p_pool * (1 - p_pool) * 2 / n)
    z = (a.mean() - b.mean()) / se if se > 0 else 0
    p_values.append(2 * (1 - stats.norm.cdf(abs(z))))

plt.figure(figsize=(12, 4))
plt.plot(list(check_points), p_values, 'b-', alpha=0.7)
plt.axhline(0.05, color='red', linestyle='--', label='α = 0.05')
plt.fill_between(list(check_points), 0, 0.05, alpha=0.1, color='red')
plt.xlabel('Sample Size per Group')
plt.ylabel('p-value')
plt.title('Peeking Problem: p-value Fluctuates Over Time (NO real effect!)')
plt.legend()
plt.tight_layout()
plt.show()

false_positives = sum(1 for p in p_values if p < 0.05)
print(f"Times p < 0.05: {false_positives}/{len(p_values)} ({false_positives/len(p_values)*100:.0f}%)")
print("If you stop at any of those points, you would declare a 'winner' that doesn't exist!")

---
## 4. One-Way ANOVA — Comparing 3+ Groups

When you have **3 or more groups**, you cannot just run multiple t-tests (that inflates Type I error). ANOVA tests whether *any* group mean differs from the others.

In [ ]:
## One-Way ANOVA
# Question: Do 3 teaching methods produce different exam scores?

np.random.seed(42)
method_a = np.random.normal(75, 10, 30)   # Traditional
method_b = np.random.normal(80, 10, 30)   # Blended
method_c = np.random.normal(78, 10, 30)   # Online

# H₀: μA = μB = μC (all means equal)
# H₁: At least one mean is different

f_stat, p_value = stats.f_oneway(method_a, method_b, method_c)

print(f"Method A mean: {method_a.mean():.1f}")
print(f"Method B mean: {method_b.mean():.1f}")
print(f"Method C mean: {method_c.mean():.1f}")
print(f"\nF-statistic  : {f_stat:.3f}")
print(f"p-value      : {p_value:.4f}")
print(f"Decision     : {'At least one method differs' if p_value < 0.05 else 'No significant difference'}")

# ANOVA tells you SOMETHING is different, but not WHICH groups.
# For that, you need post-hoc tests (next cell).

In [ ]:
## Post-hoc test: Tukey HSD — which pairs differ?

from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Combine all data with labels
all_scores = np.concatenate([method_a, method_b, method_c])
labels = ['Traditional']*30 + ['Blended']*30 + ['Online']*30

tukey = pairwise_tukeyhsd(all_scores, labels, alpha=0.05)
print(tukey)

# Tukey corrects for multiple comparisons. If you ran 3 separate t-tests,
# your overall α would be ~14% instead of 5%. Tukey keeps it at 5%.

---
## 5. Repeated Measures and DOE (Design of Experiments)

| Design | What It Is | Example |
|---|---|---|
| **Between-subjects** | Different people in each group | Standard A/B test |
| **Within-subjects (Repeated)** | Same people in all conditions | Before/after training |
| **Factorial (2×2, 2×3)** | Test multiple factors simultaneously | Button color × text length |
| **Randomized Block** | Group similar subjects, randomize within blocks | Test by age group |
| **Crossover** | Each subject gets both treatments (in random order) | Drug A then B vs B then A |

### Why Factorial Designs Are Powerful
Instead of running two separate A/B tests (one for color, one for text), a 2×2 factorial tests **both factors AND their interaction** with the same sample size.

In [ ]:
## Two-way ANOVA — factorial design example

import pandas as pd
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

np.random.seed(42)
n = 20

# 2×2 Factorial: Button Color (Red/Blue) × CTA Text (Buy/Get)
data = pd.DataFrame({
    'color': ['Red']*n*2 + ['Blue']*n*2,
    'text':  (['Buy']*n + ['Get']*n) * 2,
    'clicks': np.concatenate([
        np.random.normal(50, 10, n),    # Red + Buy
        np.random.normal(55, 10, n),    # Red + Get
        np.random.normal(48, 10, n),    # Blue + Buy
        np.random.normal(60, 10, n),    # Blue + Get (interaction!)
    ])
})

# Fit ANOVA model with interaction term
model = ols('clicks ~ C(color) * C(text)', data=data).fit()
anova_table = anova_lm(model, typ=2)

print("=== Two-Way ANOVA ===")
print(anova_table.round(4))
print("\nLook at p-values: color, text, and their interaction (color:text).")
print("A significant interaction means the effect of text depends on color.")

---
## Key Libraries and Packages

| Library | Focus Area |
|---|---|
| **`scipy.stats`** | Core hypothesis tests (t, Z, chi², ANOVA, etc.) |
| **`statsmodels`** | Regression, ANOVA tables, power analysis, time series |
| **`pingouin`** | Friendly stats API (effect sizes, Bayesian tests, mixed ANOVA) |
| **`scikit-posthocs`** | Post-hoc tests beyond Tukey (Dunn, Nemenyi, etc.) |
| **`pymc`** | Bayesian statistics (probabilistic programming) |
| **`lifelines`** | Survival analysis (Kaplan-Meier, Cox regression) |

> **Perspective:** Frequentist A/B testing (p-values, power) is the industry standard, but **Bayesian A/B testing** is gaining traction. Bayesian methods give you P(B > A) directly (e.g., "92% chance the new page is better"), which is more intuitive for business stakeholders than "p = 0.03". Libraries like `pymc` and services like Google Optimize use Bayesian approaches.

---
## A/B Testing Checklist

1. **Define** your metric (conversion rate, revenue per user, etc.)
2. **Calculate** sample size using power analysis BEFORE starting
3. **Randomize** users into control and treatment groups
4. **Run** the experiment for the planned duration — **do not peek**
5. **Analyze** using the appropriate test (Z-test for proportions, t-test for means)
6. **Report** effect size + confidence interval, not just p-value
7. **Consider** practical significance — a statistically significant 0.01% lift may not be worth the engineering cost

> **Final Thought:** Statistical significance ≠ practical significance. A test might prove a 0.1% lift is real (p < 0.05), but if it requires 6 months of engineering to ship, it is not worth it. Always pair statistical analysis with business judgment.